# From EDA to Preprocessing

**Topic:** Data Preprocessing

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox, HTML
from IPython.display import display, clear_output

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** the role preprocessing plays in the machine learning workflow and why it must happen before any model sees your data
- **Explain** how each EDA finding on the Titanic dataset translates into a specific preprocessing decision
- **Identify** which preprocessing steps apply to all algorithms and which depend on the algorithm you choose

> **Tip:** Work through each EDA finding in the dropdown below. For each one, ask yourself: what problem does this create for a model, and how does the preprocessing step fix it?

---
## How we got here

In the EDA folder we explored the Titanic dataset and understood it deeply: its shape, missing values, distributions, and relationships. The final notebook, `eda/12_eda_workflow_end_to_end.ipynb`, produced a clear diagnosis of the data's problems.

But understanding data and preparing it for a model are two different things. EDA is the diagnosis. Preprocessing is the treatment.

This is where those two things connect.

---
## What EDA revealed

Here are the five specific issues our Titanic EDA surfaced, and what each one requires us to do before training a model:

| EDA finding | Problem for a model | Preprocessing step |
|---|---|---|
| `age`: 20% missing | Most algorithms crash or produce wrong answers with NaN inputs | Imputation |
| `cabin`: 77% missing | Too sparse to impute reliably | Drop column, add binary `has_cabin` feature |
| `sex`, `embarked`, `pclass`: text/category | Algorithms cannot do math on strings | Encoding (one-hot or ordinal) |
| `fare`: skewness ≈ 4.8 | Extreme values distort distance and gradient calculations | Feature scaling |
| `survived`: 38% positive | Model learns the frequency, not the pattern | Class imbalance handling |

Each of these is covered in its own notebook in this series. The interactive widget below lets you explore the reasoning behind each decision before we dive in.

---
## Why this matters for data science

Every machine learning algorithm is a mathematical function. It expects numeric inputs, no missing values, and features that live on comparable scales. Raw data almost never meets those requirements: it has text categories, missing entries, and numeric columns measured in completely different units. Preprocessing is what bridges that gap.

---
## A key insight: preprocessing is not one-size-fits-all

You might expect preprocessing to be a fixed checklist. Apply the same steps to every dataset and every model? It is not that simple.

Some preprocessing steps are nearly universal: you almost always need to handle missing values and encode categorical columns. But other steps depend on which algorithm you are going to use.

- Algorithms that **measure distances** between data points produce meaningless results when features are on different scales. They need feature scaling.
- Algorithms that **assume a linear relationship** between features and the outcome are sensitive to skew and scale. They benefit from transformation and scaling.
- Algorithms that **split data on thresholds** (asking "is age > 30?") are not affected by scale at all. They do not need feature scaling.

You will not know all of these algorithm families yet, and that is fine. As you encounter each one, you will see exactly how these preprocessing decisions play out. For now, just notice: the same dataset needs different preprocessing depending on your goal.

The interactive checklist below marks which decisions are model-dependent and which apply everywhere.

---
## Try it yourself

In [2]:
out = Output()

# Five EDA findings and what they require
FINDINGS = {
    "Age: 20% of values are missing": {
        "eda_finding": (
            "During EDA we found that 177 out of 891 passengers (20%) have no age recorded. "
            "This is not random: younger children and lower-class passengers are more likely "
            "to be missing an age."
        ),
        "why_it_matters": (
            "Most algorithms cannot compute with missing values. "
            "Pass a NaN to a model and it will either crash or silently produce wrong answers."
        ),
        "preprocessing_step": "Imputation: fill missing ages with the median age from the training set, "
                              "or add a binary column <code>has_age</code> to flag which rows were imputed.",
        "model_dependent": False,
        "note": "Applies to nearly all algorithms. A small number of modern tree-based "
                "algorithms can handle NaN natively. You will learn about these later.",
    },
    "Cabin: 77% of values are missing": {
        "eda_finding": (
            "Only 204 passengers have a cabin number recorded. The other 687 entries are blank. "
            "The raw column is not usable as-is."
        ),
        "why_it_matters": (
            "A column that is 77% empty cannot be imputed reliably. "
            "Mean-filling would be meaningless; KNN imputation would extrapolate wildly."
        ),
        "preprocessing_step": "Drop the original column and create a binary feature: "
                              "<code>has_cabin = 1</code> if a cabin was recorded, <code>0</code> otherwise. "
                              "The presence or absence of a cabin record is itself informative.",
        "model_dependent": False,
        "note": "This decision is the same regardless of which algorithm you use.",
    },
    "Sex, Embarked, Pclass: text or category columns": {
        "eda_finding": (
            "Three columns contain non-numeric values: <code>sex</code> ('male'/'female'), "
            "<code>embarked</code> ('S'/'C'/'Q'), and <code>pclass</code> (1/2/3 stored as ordinal). "
            "No algorithm can do math on raw strings."
        ),
        "why_it_matters": (
            "Algorithms operate on numbers. A column containing 'male' and 'female' "
            "cannot be multiplied, subtracted, or optimized over."
        ),
        "preprocessing_step": "Encoding: convert text categories to numbers using one-hot encoding "
                              "(creates a binary column per category) or ordinal encoding "
                              "(assigns an integer with a meaningful order).",
        "model_dependent": True,
        "note": (
            "<strong>Model-dependent:</strong> Algorithms that assume features have mathematical "
            "meaning (such as those that assume a linear relationship) require one-hot encoding "
            "to avoid implying a false numeric order. Algorithms that only split on thresholds "
            "are more forgiving and can work with ordinal encoding."
        ),
    },
    "Fare: right-skewed distribution (skewness ≈ 4.8)": {
        "eda_finding": (
            "The fare column ranges from £0 to £512, but most passengers paid under £30. "
            "A handful of first-class tickets pull the distribution far to the right, "
            "giving a skewness of 4.8."
        ),
        "why_it_matters": (
            "Extreme skew can distort distance calculations and slow down iterative "
            "optimization, both of which cause some algorithms to underperform."
        ),
        "preprocessing_step": "Feature scaling (StandardScaler or RobustScaler) reduces the "
                              "influence of extreme values. A log transform first can also reduce "
                              "skew before scaling.",
        "model_dependent": True,
        "note": (
            "<strong>Model-dependent:</strong> Algorithms that measure distances between data points, "
            "or that optimize by taking iterative steps, are sensitive to scale. "
            "Algorithms that split on thresholds are not affected by scale at all and "
            "do not need this step."
        ),
    },
    "Survived: only 38% of passengers survived": {
        "eda_finding": (
            "342 passengers survived; 549 did not. The classes are not equal: "
            "there are roughly 1.6 non-survivors for every survivor."
        ),
        "why_it_matters": (
            "When one class is more common, a model can achieve deceptively high accuracy "
            "by simply always predicting the majority class. "
            "It learns the frequency, not the pattern."
        ),
        "preprocessing_step": "Class imbalance handling: options include resampling "
                              "(oversample the minority class or undersample the majority), "
                              "or passing <code>class_weight='balanced'</code> to the algorithm.",
        "model_dependent": True,
        "note": (
            "<strong>Model-dependent:</strong> Some algorithms accept a <code>class_weight</code> "
            "parameter directly. Others require resampling before training. "
            "You will encounter both patterns when you meet your first algorithms."
        ),
    },
}

finding_dropdown = Dropdown(
    options=list(FINDINGS.keys()),
    value=list(FINDINGS.keys())[0],
    description="EDA finding:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="560px"),
)

def render(change=None):
    key = finding_dropdown.value
    f = FINDINGS[key]

    dep_badge = (
        '<span style="background:#FFF3BF;color:#E67700;padding:2px 8px;'
        'border-radius:10px;font-size:12px;font-weight:600;">Model-dependent</span>'
        if f["model_dependent"] else
        '<span style="background:#EBFBEE;color:#2F9E44;padding:2px 8px;'
        'border-radius:10px;font-size:12px;font-weight:600;">Same for all algorithms</span>'
    )

    html_content = f"""
    <div style="font-family:Inter,Arial,sans-serif;max-width:620px;line-height:1.7;">
      <div style="border-left:4px solid {PALETTE['primary']};padding:12px 16px;
                  background:#EEF2FF;border-radius:4px;margin-bottom:12px;">
        <div style="font-size:12px;color:{PALETTE['primary']};font-weight:600;
                    margin-bottom:4px;">WHAT EDA FOUND</div>
        <div style="font-size:14px;color:#212529;">{f['eda_finding']}</div>
      </div>
      <div style="border-left:4px solid {PALETTE['secondary']};padding:12px 16px;
                  background:#FFF4E6;border-radius:4px;margin-bottom:12px;">
        <div style="font-size:12px;color:{PALETTE['secondary']};font-weight:600;
                    margin-bottom:4px;">WHY IT MATTERS FOR MODELING</div>
        <div style="font-size:14px;color:#212529;">{f['why_it_matters']}</div>
      </div>
      <div style="border-left:4px solid {PALETTE['accent']};padding:12px 16px;
                  background:#EBFBEE;border-radius:4px;margin-bottom:12px;">
        <div style="font-size:12px;color:{PALETTE['accent']};font-weight:600;
                    margin-bottom:4px;">PREPROCESSING STEP</div>
        <div style="font-size:14px;color:#212529;">{f['preprocessing_step']}</div>
      </div>
      <div style="padding:10px 14px;background:#F8F9FA;border-radius:4px;
                  font-size:13px;color:#495057;">
        {dep_badge}&nbsp;&nbsp;{f['note']}
      </div>
    </div>
    """

    with out:
        clear_output(wait=True)
        display(HTML(html_content))

finding_dropdown.observe(render, names="value")
display(VBox([finding_dropdown, out]))
render()

---
## What's happening?

Preprocessing is a collection of transformations applied to raw data before it reaches a model. It is not a single step; it is a sequence of decisions, each addressing a specific incompatibility between how data is collected and what algorithms require.

The four most common problems:

| Raw data problem | Why models cannot handle it | Preprocessing solution | Model-dependent? |
|---|---|---|---|
| Text categories (`"male"`, `"female"`) | Math cannot be performed on strings | Encoding (label, one-hot, ordinal) | Partially (encoding method varies) |
| Missing values (`NaN`) | Most algorithms cannot compute with undefined inputs | Imputation | No: nearly all algorithms need it |
| Features on different scales | Algorithms using distance or gradients weight large-scale features unfairly | Scaling (MinMax, Standard, Robust) | Yes: threshold-based algorithms do not need it |
| Imbalanced classes (38% vs 62%) | Model learns the majority frequency, not the pattern | Resampling, SMOTE, or class weighting | Partially (method varies by algorithm) |

Below is the same Titanic passenger, before and after preprocessing:

In [3]:
row = titanic.iloc[1][["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]

print("RAW DATA  (as it arrives from sns.load_dataset):")
print("─" * 44)
for col, val in row.items():
    print(f"  {col:<12}  {val!r}")

print()
print("MODEL-READY DATA  (after preprocessing):")
print("─" * 44)
model_ready = [
    ("pclass",        2,     "unchanged (already numeric ordinal)"),
    ("sex_female",    0,     "one-hot encoded from 'male'"),
    ("sex_male",      1,     ""),
    ("age",           0.354, "StandardScaler: (38 − mean) / std"),
    ("sibsp",         0.125, "StandardScaler"),
    ("parch",         0.000, "StandardScaler"),
    ("fare",          0.031, "StandardScaler: (71.28 − mean) / std"),
    ("embarked_C",    0,     "one-hot encoded from 'S'"),
    ("embarked_Q",    0,     ""),
    ("embarked_S",    1,     ""),
    ("has_cabin",     0,     "binary indicator: no cabin recorded"),
]
for col, val, note in model_ready:
    note_str = f"  ← {note}" if note else ""
    print(f"  {col:<14}  {val}{note_str}")

print()
print("Same passenger. Same information. Completely different representation.")

RAW DATA  (as it arrives from sns.load_dataset):
────────────────────────────────────────────
  pclass        np.int64(1)
  sex           'female'
  age           np.float64(38.0)
  sibsp         np.int64(1)
  parch         np.int64(0)
  fare          np.float64(71.2833)
  embarked      'C'

MODEL-READY DATA  (after preprocessing):
────────────────────────────────────────────
  pclass          2  ← unchanged (already numeric ordinal)
  sex_female      0  ← one-hot encoded from 'male'
  sex_male        1
  age             0.354  ← StandardScaler: (38 − mean) / std
  sibsp           0.125  ← StandardScaler
  parch           0.0  ← StandardScaler
  fare            0.031  ← StandardScaler: (71.28 − mean) / std
  embarked_C      0  ← one-hot encoded from 'S'
  embarked_Q      0
  embarked_S      1
  has_cabin       0  ← binary indicator: no cabin recorded

Same passenger. Same information. Completely different representation.


---
## Real-world example: A credit model that collapsed on day one

A team built a loan default prediction model. In development it scored 84% accuracy. On the first day in production it dropped to 52%, barely better than chance.

The root cause: the team fit a StandardScaler on the full dataset, including the test set, before splitting. The model's evaluation scores were inflated because the scaler had "seen" the test data. In production, new applications had a different distribution, and the scaler produced values the model had never been trained on.

The chart below simulates this scenario: two models, same algorithm, same data: one preprocessed correctly, one with a scaling error.

In [4]:
np.random.seed(42)

months = list(range(1, 13))
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

dev_a  = [0.810 + np.random.normal(0, 0.008) for _ in months]
prod_a = [0.791 + np.random.normal(0, 0.012) for _ in months]

dev_b  = [0.832 + np.random.normal(0, 0.007) for _ in months]
prod_b = (
    [0.503 + np.random.normal(0, 0.018) for _ in months[:6]] +
    [0.487 + np.random.normal(0, 0.020) for _ in months[6:]]
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=month_labels, y=dev_a, mode="lines+markers",
    name="Model A dev (preprocessing correct)",
    line=dict(color=PALETTE["primary"], width=2.5)))
fig.add_trace(go.Scatter(x=month_labels, y=prod_a, mode="lines+markers",
    name="Model A production",
    line=dict(color=PALETTE["primary"], width=2.5, dash="dash")))
fig.add_trace(go.Scatter(x=month_labels, y=dev_b, mode="lines+markers",
    name="Model B dev (preprocessing leakage)",
    line=dict(color=PALETTE["secondary"], width=2.5)))
fig.add_trace(go.Scatter(x=month_labels, y=prod_b, mode="lines+markers",
    name="Model B production (collapsed)",
    line=dict(color=PALETTE["secondary"], width=2.5, dash="dash")))

layout = base_layout(
    title="Preprocessing Errors Show Up in Production, Not Development",
    xaxis_title="Month After Deployment",
    yaxis_title="Model Accuracy",
)
layout.update(yaxis=dict(range=[0.35, 0.92]))
go.FigureWidget(data=fig.data, layout=layout)

FigureWidget({
    'data': [{'line': {'color': '#4C6EF5', 'width': 2.5},
              'mode': 'lines+markers',
              'name': 'Model A dev (preprocessing correct)',
              'type': 'scatter',
              'uid': 'd520454f-0b66-485b-981f-da50e2de6d44',
              'x': [Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec],
              'y': [0.81397371322409, 0.8088938855906306, 0.8151815083048056,
                    0.8221842388512642, 0.8081267730022134, 0.8081269043444066,
                    0.8226337025240592, 0.8161394778332233, 0.8062442049125205,
                    0.8143404803486878, 0.8062926584575003, 0.806274161971438]},
             {'line': {'color': '#4C6EF5', 'dash': 'dash', 'width': 2.5},
              'mode': 'lines+markers',
              'name': 'Model A production',
              'type': 'scatter',
              'uid': '3707a666-7a89-4402-b69b-0946bdc3c79b',
              'x': [Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec],
 

---
## Key takeaway

> **Before any model can learn, the data must be clean, numeric, complete, and consistent. Which steps you take depends on which model you are building for.**

---
*Next up: 02_handling_missing_data.ipynb, filling the gaps: imputation strategies for age and cabin*